In [1]:
/**
 * @file grafos_jupyter_safe_documentado.cpp
 * @brief Ecosistema de Grafos: Representación, Recorridos y Benchmarking .
 * Este código unifica las estructuras fundamentales (Matriz y Lista de Adyacencia)
 * con los motores algorítmicos (BFS, DFS) descritos en el Capítulo 20 de Cormen et al.
 * y el Capítulo 8 de Mahdi. 
 */

#include <iostream>
#include <vector>
#include <cstdint>
#include <stdexcept>
#include <algorithm>
#include <chrono>
#include <memory>
#include <iomanip>
#include <functional>
#include <queue>
#include <limits>

// ============================================================================
// 1. CONTRATO DE INTERFAZ (POLIMORFISMO ABSTRACTO)
// ============================================================================

/**
 * @class IGraph
 * @brief Interfaz base estricta que define el comportamiento topológico de cualquier grafo.
 * Al usar una interfaz abstracta, desacoplamos los algoritmos (BFS/DFS) de la 
 * estructura de memoria subyacente (Matriz o Lista).
 */
class IGraph {
public:
    // Destructor virtual: Crítico. Garantiza que cuando un puntero IGraph* // sea destruido, se llame al destructor de la clase derivada correcta,
    // previniendo fugas de memoria (memory leaks).
    virtual ~IGraph() = default;

    // Conecta el vértice 'u' con el vértice 'v'.
    virtual void addEdge(uint32_t u, uint32_t v) = 0;

    // Verifica la existencia de la arista (u, v) en tiempo de ejecución.
    // El sufijo 'const' garantiza que este método es de solo lectura.
    virtual bool hasEdge(uint32_t u, uint32_t v) const = 0;

    // Extrae todos los vértices adyacentes a 'u'.
    virtual std::vector<uint32_t> getNeighbors(uint32_t u) const = 0;

    // Retorna la cardinalidad del conjunto de vértices |V|.
    // 'noexcept' asegura al compilador que esta operación no lanzará excepciones,
    // permitiendo optimizaciones agresivas en tiempo de compilación.
    virtual uint32_t getVertexCount() const noexcept = 0;
};

// ============================================================================
// 2. ESTRUCTURAS SUBYACENTES DE MEMORIA
// ============================================================================

/**
 * @class AdjacencyMatrix
 * @brief Implementación para Grafos Densos. Complejidad Espacial: O(V^2).
 */
class AdjacencyMatrix : public IGraph {
private:
    uint32_t numVertices;
    
    // Matriz de adyacencia.
    // Nota de Diseño: std::vector<bool> en C++ no almacena booleanos estándar (8 bits).
    // Implementa una especialización de "Bit-Packing", guardando 8 valores en un solo byte.
    std::vector<std::vector<bool>> matrix;

    // Función interna (inline para evitar overhead de llamada a función) para validar índices.
    inline void validateVertex(uint32_t v) const {
        if (v >= numVertices) throw std::out_of_range("Fallo de Segmentación: Vértice inexistente.");
    }

public:
    // Constructor explícito (evita conversiones de tipo accidentales).
    explicit AdjacencyMatrix(uint32_t vertices) : numVertices(vertices) {
        // Pre-asigna memoria para una matriz de VxV inicializada en false.
        matrix.resize(numVertices, std::vector<bool>(numVertices, false));
    }

    // Inserción: Tiempo O(1) puro.
    void addEdge(uint32_t u, uint32_t v) override {
        validateVertex(u); validateVertex(v);
        matrix[u][v] = true;
        matrix[v][u] = true; // Simetría asume grafo no dirigido.
    }

    // Consulta de Arista: Tiempo O(1) puro. Esta es la mayor fortaleza de la Matriz.
    bool hasEdge(uint32_t u, uint32_t v) const override {
        validateVertex(u); validateVertex(v);
        return matrix[u][v];
    }

    // Obtención de Adyacentes: Tiempo O(V). 
    // Deficiencia Arquitectónica: Para encontrar los vecinos, la matriz DEBE iterar
    // a través de TODOS los 'V' vértices en la fila, escaneando miles de ceros inútiles.
    std::vector<uint32_t> getNeighbors(uint32_t u) const override {
        validateVertex(u);
        std::vector<uint32_t> neighbors;
        neighbors.reserve(numVertices / 2); // Heurística para evitar reasignaciones dinámicas del vector.
        
        for (uint32_t v = 0; v < numVertices; ++v) {
            if (matrix[u][v]) {
                neighbors.push_back(v);
            }
        }
        return neighbors;
    }

    uint32_t getVertexCount() const noexcept override { return numVertices; }
};

/**
 * @class AdjacencyList
 * @brief Implementación para Grafos Dispersos. Complejidad Espacial: O(V + E).
 */
class AdjacencyList : public IGraph {
private:
    uint32_t numVertices;
    
    // Lista de adyacencia usando el patrón Structure of Arrays (Arreglo de Arreglos).
    // Garantiza que los vecinos de un nodo estén en bloques de memoria RAM estrictamente
    // contiguos, disparando la eficiencia de la Caché L1 del procesador durante un BFS/DFS.
    std::vector<std::vector<uint32_t>> adj;

    inline void validateVertex(uint32_t v) const {
        if (v >= numVertices) throw std::out_of_range("Fallo de Segmentación: Vértice inexistente.");
    }

public:
    explicit AdjacencyList(uint32_t vertices) : numVertices(vertices) {
        adj.resize(numVertices); 
    }

    // Inserción: Tiempo O(1) amortizado por el std::vector.
    void addEdge(uint32_t u, uint32_t v) override {
        validateVertex(u); validateVertex(v);
        adj[u].push_back(v);
        adj[v].push_back(u);
    }

    // Consulta de Arista: Tiempo O(deg(u)).
    // Deficiencia Arquitectónica: La Lista sufre aquí. Si el vértice tiene muchos vecinos,
    // hay que realizar una búsqueda secuencial hasta encontrar 'v' o agotar la lista.
    bool hasEdge(uint32_t u, uint32_t v) const override {
        validateVertex(u);
        const auto& neighbors = adj[u];
        // Utiliza algoritmo estándar de C++ para buscar en rango.
        return std::find(neighbors.begin(), neighbors.end(), v) != neighbors.end();
    }

    // Obtención de Adyacentes: Tiempo O(1) referencial.
    // Retorna una copia del vector de vecinos. En C++ moderno, la optimización NRVO 
    // (Named Return Value Optimization) evita que esta copia suceda físicamente, 
    // devolviendo la estructura casi instantáneamente.
    std::vector<uint32_t> getNeighbors(uint32_t u) const override {
        validateVertex(u);
        return adj[u];
    }

    uint32_t getVertexCount() const noexcept override { return numVertices; }
};

// ============================================================================
// 3. MOTORES DE RECORRIDO Y ANÁLISIS TOPOLÓGICO (MODELO CORMEN)
// ============================================================================

// Constantes globales de estado para la Máquina de Estados de Recorrido.
constexpr uint8_t WHITE = 0; // Vértice no descubierto.
constexpr uint8_t GRAY  = 1; // Vértice en la frontera (descubierto, no finalizado).
constexpr uint8_t BLACK = 2; // Vértice finalizado (procesado él y sus adyacencias).
constexpr int INF = std::numeric_limits<int>::max(); // Distancia infinita inicial.
constexpr int NIL = -1; // Predecesor nulo (raíz del árbol de recorrido).

// Definición de un callback para inyectar lógica de negocio en el recorrido (Patrón Visitador).
using VertexVisitor = std::function<void(uint32_t)>;

class GraphTraversals {
private:
    // Subrutina privada y recursiva para el motor DFS.
    static void DFS_Visit(const IGraph& graph, uint32_t u, int& time, 
                          std::vector<uint8_t>& color, std::vector<int>& pi, 
                          std::vector<int>& d, std::vector<int>& f, 
                          const VertexVisitor& visit) {
        
        // Fase 1: Descubrimiento (Pre-orden)
        time++;           // Avanza el reloj topológico
        d[u] = time;      // Sella la marca de tiempo de descubrimiento
        color[u] = GRAY;  // Colorea a GRIS (En evaluación)
        visit(u);         // Ejecuta la inyección de código del usuario
        
        // Fase 2: Expansión de la Rama (Profundidad LIFO)
        for (const uint32_t& v : graph.getNeighbors(u)) {
            if (color[v] == WHITE) {
                // Si es BLANCO, es una Arista de Árbol (Tree Edge). Exploramos.
                pi[v] = static_cast<int>(u); // Construye el bosque DFS
                DFS_Visit(graph, v, time, color, pi, d, f, visit);
            }
            // Nota Teórica: Si color[v] == GRAY aquí, detectamos un CICLO (Arista de Retroceso).
        }
        
        // Fase 3: Finalización (Post-orden)
        color[u] = BLACK; // Vértice exhausto
        time++;           // Avanza el reloj
        f[u] = time;      // Sella la marca de tiempo de finalización
    }

public:
    // Motor estático: Búsqueda en Anchura (BFS)
    // Garantiza el camino mínimo en grafos no ponderados. Complejidad: O(V + E)
    static void BFS(const IGraph& graph, uint32_t s, const VertexVisitor& visit) {
        uint32_t V = graph.getVertexCount();
        
        // Asignación de atributos paralelos (SoA) para optimizar memoria caché.
        std::vector<uint8_t> color(V, WHITE);
        std::vector<int> d(V, INF);
        std::vector<int> pi(V, NIL);
        
        // Estructura obligatoria para expansión por niveles
        std::queue<uint32_t> Q;
        
        // Inicialización del nodo origen
        color[s] = GRAY; 
        d[s] = 0; 
        Q.push(s);
        
        // Bucle expansivo temporal (onda de propagación)
        while (!Q.empty()) {
            uint32_t u = Q.front(); // Extrae el más antiguo de la frontera (FIFO)
            Q.pop();
            visit(u); 
            
            for (const uint32_t& v : graph.getNeighbors(u)) {
                if (color[v] == WHITE) {
                    // Descubrimiento en la periferia
                    color[v] = GRAY; 
                    d[v] = d[u] + 1; // La distancia es el padre + 1 arista
                    pi[v] = static_cast<int>(u);
                    Q.push(v);       // Encola para la siguiente capa de expansión
                }
            }
            color[u] = BLACK; // Termina con este vértice
        }
    }

    // Motor estático: Búsqueda en Profundidad (DFS)
    // Garantiza exploración exhaustiva y topológica. Complejidad: O(V + E)
    static void DFS(const IGraph& graph, const VertexVisitor& visit) {
        uint32_t V = graph.getVertexCount();
        
        // Arreglos paralelos de estado topológico
        std::vector<uint8_t> color(V, WHITE);
        std::vector<int> pi(V, NIL);
        std::vector<int> d(V, 0); // Descubrimiento
        std::vector<int> f(V, 0); // Finalización
        
        int time = 0; // Reloj topológico global, pasado por referencia
        
        // Bucle externo vital: Asegura procesar nodos aislados en grafos disconexos.
        for (uint32_t u = 0; u < V; ++u) {
            if (color[u] == WHITE) {
                DFS_Visit(graph, u, time, color, pi, d, f, visit);
            }
        }
    }
};

// ============================================================================
// 4. APLICACIONES AVANZADAS
// ============================================================================

class GraphApplications {
public:
    // Determina si un grafo puede ser dividido en dos conjuntos disjuntos U y W
    // de tal forma que ninguna arista conecte nodos dentro del mismo conjunto.
    // Complejidad: O(V + E)
    static bool isBipartite(const IGraph& graph, uint32_t startNode) {
        uint32_t V = graph.getVertexCount();
        
        // Vector de colores (0 para conjunto U, 1 para conjunto W, -1 no visitado)
        std::vector<int> color(V, -1);
        std::queue<uint32_t> Q;

        // Arrancamos asumiendo que el origen pertenece al conjunto 0
        color[startNode] = 0; 
        Q.push(startNode);

        // Expansión BFS (ideal para colorear capas alternas)
        while (!Q.empty()) {
            uint32_t u = Q.front(); 
            Q.pop();

            for (const uint32_t& v : graph.getNeighbors(u)) {
                if (color[v] == -1) {
                    // Si el vecino es virgen, lo forzamos al conjunto contrario (1 - 0 = 1)
                    color[v] = 1 - color[u]; 
                    Q.push(v);
                } 
                else if (color[v] == color[u]) {
                    // FALLO CRÍTICO: Una arista conecta dos nodos del MISMO conjunto.
                    // Teorema: Se ha detectado un ciclo de longitud impar. No es bipartito.
                    return false; 
                }
            }
        }
        return true;
    }
};

// ============================================================================
// 5. MÓDULO DE BENCHMARKING FÍSICO
// ============================================================================

class GraphBenchmark {
private:
    // Evalúa el costo de extraer las listas de vecinos (Cuello de botella de Memoria)
    static double measureNeighborsTraversal(const IGraph& graph) {
        auto start = std::chrono::high_resolution_clock::now();
        uint32_t V = graph.getVertexCount();
        
        // La directiva 'volatile' previene que el compilador agresivo (o el intérprete JIT)
        // elimine este bucle entero al darse cuenta de que 'dummyCount' no se usa.
        volatile size_t dummyCount = 0; 
        
        // Emulación de la carga de trabajo de un BFS/DFS
        for (uint32_t i = 0; i < V; ++i) {
            auto neighbors = graph.getNeighbors(i);
            dummyCount += neighbors.size();
        }

        auto end = std::chrono::high_resolution_clock::now();
        std::chrono::duration<double, std::milli> diff = end - start;
        return diff.count(); // Retorna el tiempo en milisegundos reales.
    }

public:
    static void runSparseGraphTest() {
        std::cout << "\n[PRUEBA 1] GRAFO DISPERSO (Ej. Mapas, Moléculas)\n";
        // V=500 es el límite seguro para que el JIT de Jupyter no cancele por Timeout.
        constexpr uint32_t V = 500; 
        
        AdjacencyMatrix mat(V); 
        AdjacencyList lst(V);

        // Topología: Cada nodo se conecta a sus dos vecinos siguientes (Anillo disperso).
        for (uint32_t i = 0; i < V; ++i) {
            mat.addEdge(i, (i + 1) % V); mat.addEdge(i, (i + 2) % V);
            lst.addEdge(i, (i + 1) % V); lst.addEdge(i, (i + 2) % V);
        }

        std::cout << "Midiendo operación getNeighbors() en topología de baja densidad (V=" << V << ")...\n";
        std::cout << " -> Tiempo MATRIZ : " << std::fixed << std::setprecision(3) << measureNeighborsTraversal(mat) << " ms\n";
        std::cout << " -> Tiempo LISTA  : " << std::fixed << std::setprecision(3) << measureNeighborsTraversal(lst) << " ms\n";
        // Conclusión esperada: La Lista aplasta a la Matriz porque la Matriz escanea 500 celdas por fila
        // para encontrar solo 2 aristas reales.
    }

    static void runDenseGraphTest() {
        std::cout << "\n[PRUEBA 2] GRAFO DENSO (Ej. Red LAN Totalmente Conectada)\n";
        constexpr uint32_t V = 100;

        AdjacencyMatrix mat(V); 
        AdjacencyList lst(V);

        // Topología Completa (Kn): Todos los nodos se conectan con todos (V * (V-1) / 2 aristas).
        for (uint32_t i = 0; i < V; ++i) {
            for (uint32_t j = i + 1; j < V; ++j) {
                mat.addEdge(i, j); 
                lst.addEdge(i, j);
            }
        }

        std::cout << "Midiendo operación hasEdge() en topología de máxima densidad (10,000 queries)...\n";
        
        // Benchmark para Matriz O(1)
        auto startMat = std::chrono::high_resolution_clock::now();
        volatile size_t hitsMat = 0;
        for(uint32_t k = 0; k < 10000; ++k) { 
            if(mat.hasEdge(k % V, (k + 50) % V)) hitsMat++; 
        }
        auto endMat = std::chrono::high_resolution_clock::now();

        // Benchmark para Lista O(deg)
        auto startLst = std::chrono::high_resolution_clock::now();
        volatile size_t hitsLst = 0;
        for(uint32_t k = 0; k < 10000; ++k) { 
            if(lst.hasEdge(k % V, (k + 50) % V)) hitsLst++; 
        }
        auto endLst = std::chrono::high_resolution_clock::now();

        std::chrono::duration<double, std::milli> diffMat = endMat - startMat;
        std::chrono::duration<double, std::milli> diffLst = endLst - startLst;

        std::cout << " -> Tiempo MATRIZ : " << std::fixed << std::setprecision(3) << diffMat.count() << " ms\n";
        std::cout << " -> Tiempo LISTA  : " << std::fixed << std::setprecision(3) << diffLst.count() << " ms\n";
        // Conclusión esperada: La Matriz gana. La Lista tiene que iterar sobre vectores de 100 elementos
        // en cada consulta, mientras que la Matriz realiza un salto de memoria en O(1) directo.
    }
};

// ============================================================================
// 6. RUTINA PRINCIPAL DE EJECUCIÓN
// ============================================================================

void test() {
    std::cout << "========================================================\n";
    std::cout << "    EVALUACION ESTRUCTURAL DE GRAFOS (Jupyter Safe)     \n";
    std::cout << "========================================================\n";
    
    try {
        GraphBenchmark::runSparseGraphTest();
        GraphBenchmark::runDenseGraphTest();
        
        std::cout << "\n========================================================\n";
        std::cout << "    EVALUACION DE MOTORES TOPOLOGICOS (CORMEN)          \n";
        std::cout << "========================================================\n";
        
        // Se construye un grafo cíclico de 6 vértices
        AdjacencyList testGraph(6);
        testGraph.addEdge(0, 1); testGraph.addEdge(1, 2); 
        testGraph.addEdge(2, 3); testGraph.addEdge(3, 4); 
        testGraph.addEdge(4, 5); testGraph.addEdge(5, 0);

        std::cout << "Ejecutando DFS Topológico...\n";
        // Se inyecta un lambda vacío como visitor para no saturar la salida de consola
        VertexVisitor nullVisitor = [](uint32_t v) { };
        GraphTraversals::DFS(testGraph, nullVisitor);
        std::cout << "DFS ejecutado exitosamente. No hubo desbordamientos de pila.\n";

        std::cout << "\nVerificando Propiedad Bipartita...\n";
        // Un anillo de 6 nodos tiene longitud par, por lo tanto, ES bipartito matemáticamente.
        bool isBipartite = GraphApplications::isBipartite(testGraph, 0);
        std::cout << "El grafo de prueba " << (isBipartite ? "ES Bipartito." : "NO ES Bipartito.") << "\n";
        
    } catch (const std::exception& e) {
        std::cerr << "Excepción crítica interceptada: " << e.what() << '\n';
    }
}

// Invocación en tiempo de carga para el intérprete
test();

    EVALUACION ESTRUCTURAL DE GRAFOS (Jupyter Safe)     

[PRUEBA 1] GRAFO DISPERSO (Ej. Mapas, Moléculas)
Midiendo operación getNeighbors() en topología de baja densidad (V=500)...
 -> Tiempo MATRIZ : 3.706 ms
 -> Tiempo LISTA  : 0.074 ms

[PRUEBA 2] GRAFO DENSO (Ej. Red LAN Totalmente Conectada)
Midiendo operación hasEdge() en topología de máxima densidad (10,000 queries)...
 -> Tiempo MATRIZ : 0.237 ms
 -> Tiempo LISTA  : 3.426 ms

    EVALUACION DE MOTORES TOPOLOGICOS (CORMEN)          
Ejecutando DFS Topológico...
DFS ejecutado exitosamente. No hubo desbordamientos de pila.

Verificando Propiedad Bipartita...
El grafo de prueba ES Bipartito.
